# Lecture 7 - Quiz, Matrix Operations, and Solving Systems of Equations

We will start with a quiz based off of the material from homework 3. All students will have 10 minutes to complete the quiz, and the quiz will start promptly at 9:05 AM at the start of class. Please bring a pencil and paper for the exam.

We will then move to using linear algebra techniques to solve systems of equations. Linear algebra techniques, or solving an equation of the form
$$
{\bf A} \cdot {\bf x} = {\bf b},
$$
given the matrix ${\bf A}$ and vector ${\bf b}$, for the vector ${\bf x}$, lies at the heart of many numerical applications:
- Fitting a model to data
- Finding the solution to a system of equations, both linear and non-linear
- Numerical diffusion
- Partial differential equations
- Stiff ordinary equations and reaction networks (both nuclear and chemical)
- Machine learning

Today, we will focus on vector and matrix operations and algebra, and ways to solve linear sets of equations

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-7/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

This lecture made use of material from the following sources:
https://zingale.github.io/computational_astrophysics/basics/linear-algebra/la-basics.html \
https://zingale.github.io/computational_astrophysics/basics/linear-algebra/linear-system-example.html \
https://zingale.github.io/computational_astrophysics/basics/linear-algebra/gaussian_elimination.html \
https://zingale.github.io/computational_astrophysics/basics/linear-algebra/gaussian_elimination_python.html \


In [ ]:
# Nice things to have for this lecture

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Matrix Properties

There are a lot of situations that we'll see lead to needing to solve a linear system of the form:

$${\bf A}  \cdot {\bf x} = {\bf b}$$

where ${\bf A}$ is a matrix, ${\bf b}$ is a known vector, and ${\bf x}$ is the unknown vector we want to find.

We'll start by looking at general methods, but there are lots of specialized ways to solve a linear system depending on the properties of the matrix ${\bf A}$.

**Note:** There are a lot of ways to mistep thing in numerical linear algebra. See this reference for a summary:\
[Seven Sins of Numerical Linear Algebra](https://nhigham.com/2022/10/11/seven-sins-of-numerical-linear-algebra/)

Before we look at methods for solving linear systems, we'll start with reviewing some of the methods we learned in the past.

## Matrix-vector multiplication

Consider multiplying matrix ${\bf A}$ and vector ${\bf x}$ as ${\bf A}{\bf x} = {\bf b}$.  Take:

* ${\bf A}$ to be an $m\times n$ matrix
* ${\bf x}$ to be an $n\times 1$ (column) vector
* ${\bf b}$ will be an $m\times 1$ (column) vector

The multiplication looks like:

$$b_i = (A x)_i = \sum_{j=1}^M A_{ij} x_j$$

Python has it's own matrix multiplication operator, `@`, that works with NumPy arrays, so an example of this operation would be

In [ ]:
A = np.arange(12).reshape(4, 3)
A

In [ ]:
x = np.array([1, -1, 1])
A @ x

Here we see `A` is a 4x3 matrix and x is a vector of 3 elements, so the result is a 4 element vector.

Instead of using `@`, we can explicitly write out the multiplication ourselves to see the operations:

In [ ]:
b = np.zeros(A.shape[0])
for i in range(A.shape[0]):       # loop over rows
    for j in range(A.shape[1]):   # loop over columns
        b[i] += A[i, j] * x[j]

b

We see that there are 2 loops in our implementation.  This means that for a square matrix of size NxN, the number of multiplications scales as $\mathcal{O}(N^2)$

**Note:** This $\mathcal{O}(N^2)$ has a *different* meaning than the truncation error notation $\mathcal{O}(h)$, and has to do with what is called the *complexity* of this algorithm. We will discuss computational complexity next lecture. I did not create these notations, but we gotta live by them...

## Matrix-matrix multiplication

Now we can consider the case of multiplying 2 matrices ${\bf C} = {\bf A}{\bf B}$.  Now we essentially do a dot product of each row in $A$ with each column of $B$.  This looks like:

$$C_{ij} = (AB)_{ij} = \sum_{k=1}^N A_{ik} B_{kj}$$

Again, we can use the python `@` operator:

In [ ]:
B = np.array([[1, -1, 2, 3, 0],
              [2, -2, 1, 5, -7],
              [-1, 3, 4, -8, 3]])

A @ B

Now we see that `A` is a 4x3 matrix and `B` is a 3x5 matrix, so the result is a 4x5 matrix.  

Again, we can explicitly write this ourselves to see the details:

In [ ]:
assert A.shape[1] == B.shape[0]
C = np.zeros((A.shape[0], B.shape[1]))
for i in range(C.shape[0]):           # loop over rows of C
    for j in range(C.shape[1]):       # loop over columns of C
        for k in range(A.shape[1]):   # filling element C[i,j] via inner product
            C[i, j] += A[i, k] * B[k, j]

C

Now we see that there are 3 loops, which means that for square matrices, matrix multiplication scales like $\mathcal{O}(N^3)$. Again with this $\mathcal{O}$ notation, we will discuss this later...

See the article below making matrix operations faster:\
https://en.wikipedia.org/wiki/Matrix_multiplication#Computational_complexity

## Determinant

A determinant operates on a square matrix and returns a scalar that characterizes the matrix.  For our purposes, the most important property of a determinant is that a linear system, ${\bf A}{\bf x} = {\bf b}$ is solvable only if the determinant of ${\bf A}$ is nonzero.

Some common ways to note the determinant operation are:

  * $|{\bf A}|$

  * $\mathrm{det}({\bf A})$
  
Computing the determinant for small matrices is straightforward:

$$\left | \begin{array}{cc} a & b \\
                            c & d \end{array} \right | = ad - bc$$
                            
$$\left | \begin{array}{ccc} a & b & c \\
                             d & e & f \\
                             g & h & i \end{array} \right | =
                             a \left | \begin{array}{cc} e & f \\ h & i \end{array} \right | -
                             b \left | \begin{array}{cc} d & f \\ g & i \end{array} \right | +
                             c \left | \begin{array}{cc} d & e \\ g & h \end{array} \right |$$

where the 3x3 case shown above is an example of [Laplace expansion](https://en.wikipedia.org/wiki/Laplace_expansion).

While this can be extended to larger matrices, it becomes computationally expensive, and we will see a more natural way of getting the determinant as part of solving the linear system ${\bf A} \cdot {\bf x} = {\bf b}$.

## Inverse

For a matrix ${\bf A}$, the inverse, ${\bf A}^{-1}$ is defined such that

$${\bf A}{\bf A}^{-1} = {\bf A}^{-1} {\bf A} = {\bf I}$$

From this definition, we might thing that the way to solve the linear system ${\bf A} \cdot {\bf x} = {\bf b}$ is to first compute the inverse of ${\bf A}$ and then do:

$${\bf x} = {\bf A}^{-1} {\bf b}$$

However, computing the inverse of a matrix is very computationally expensive (a.k.a this is terrible), and we'll see that there are easier ways to directly solve the linear system.

There are some situations where one does need the inverse of a matrix, for instance, the covariance matrix often used in physical cosmology. Much work is devoted to making these terrible tasks less terrible. We are going to spend more time on more efficient tasks.

## Vector norm

Given a vector ${\bf v} = \{v_1, v_2, ..., v_{N}\}$ we sometimes want a single number that represents
a measure of its size---this is a [vector norm](https://en.wikipedia.org/wiki/Norm_(mathematics)).  We usually
write this as $\| {\bf v} \|$.

There are many potential definitions.  The *p-norm* is defined as:

$$\| {\bf v} \|_p = \left ( \sum_{i=1}^N |v_i|^p \right )^{1/p}$$

Some common choices are:

* $p = 1$:

  $$\| {\bf v} \|_1 =  \sum_{i=1}^N |v_i|$$
  
* $p = 2$ (Eucledian norm):

  $$\| {\bf v} \|_2 = \left ( \sum_{i=1}^N |v_i|^2 \right )^{1/2}$$

* $p = \infty$ (infinity norm):

  $$\|{\bf v}\|_\infty = \max_i |x_i|$$

Defining a [matrix norm](https://en.wikipedia.org/wiki/Matrix_norm) is more complicated. When we need to define a matrix norm for a specific application, we will do so very explicitly.

# Solving A System of Linear Equations

Consider the system:

\begin{alignat*}{4}
 x &+ y    &+ z  &= 6 \\
-x &+ 2y   &     &= 3 \\
2x &+      &+ z  &= 5
\end{alignat*}

We will solve this in steps.

1. Use the first equation to eliminate $x$ in the second two equations, resulting in:

   \begin{alignat*}{4}
    x &+ y    &+ z  &= 6 \\
      &+ 3y   &+ z  &= 9 \\
      &- 2y   &- z  &= -7
   \end{alignat*}

2. Use the second equation to eliminate $y$ in the final equation, resulting in:

   \begin{alignat*}{4}
    x &+ y    &+ z  &= 6 \\
      &+ 3y   &+ z  &= 9 \\
      &       &- \tfrac{1}{3}z  &= -1
   \end{alignat*}

   At this point, we completed _forward elimination_ &mdash; the last
   equation has a single unknown, $z$, and once we solve for it, we
   can substitute the value of $z$ in the equation above, leaving it
   with a single unknown, $y$, and so forth.

3. Now we do _back substitution_.

   From the last equation, we see $z = 3$. Putting this into the
   equation above, we get $y = 2$.  Putting both of these into the
   first equation, we finally find $x = 1$.

The process we just did is the basis of _Gaussian elimination_.

**Note 1:**
The only source of error in the procedure we did would be round-off error.
There is no truncation error.

**Note 2:**
You may have learned [Cramer's rule](https://en.wikipedia.org/wiki/Cramer's_rule)
for solving a linear system in school.  But for practical purposes, this method
is usually much more computationally expensive than Gaussian elimination.

## Matrix form

We can redo this same procedure in matrix form, starting with writing the system as:

$$
{\bf A} \cdot {\bf x} = {\bf b}
$$

with

$$
{\bf A} = \left ( \begin{array}{ccc}
                     1  &  1  &  1 \\
                    -1  &  2  &  0 \\
                     2  &  0  &  1 \end{array} \right )
$$

and

$$
{\bf b} = \left ( \begin{array}{c} 6 \\ 3 \\ 5 \end{array} \right )
$$

We will write an [augmented matrix](https://en.wikipedia.org/wiki/Augmented_matrix) $({\bf A}|{\bf b})$ as

$$
({\bf A}|{\bf b}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                    -1  &  2  &  0 & 3 \\
                     2  &  0  &  1 & 5 \end{array} \right )
$$

This allows us to easily do the same steps to both ${\bf A}$ and ${\bf b}$ together.

Doing the same sequence of steps as above, the augmented matrix looks as:

**Step 1:**
$$
({\bf A}|{\bf b}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                    -1  &  2  &  0 & 3 \\
                     2  &  0  &  1 & 5 \end{array} \right )
$$

**Step 2:**
$$
({\bf A}|{\bf b}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                     0  &  3  &  1 & 9 \\
                     0  & -2  & -1 & -7 \end{array} \right )
$$

**Step 3:**
$$
({\bf A}|{\bf b}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                     0  &  3  &  1 & 9 \\
                     0  &  0  &  -\tfrac{1}{3} & -1 \end{array} \right )
$$

At this point, the matrix is in [row echelon](https://en.wikipedia.org/wiki/Row_echelon_form), and it
clearly shows the structure needed for back-substitution.

## Pivoting

The above case worked well because all of the matrix elements were
roughly the same magnitude.  Consider the same system, with the first
term in the first equation multiplied by a small number $\epsilon$:

\begin{alignat*}{4}
 \epsilon x &+ y    &+ z  &= 6 \\
-x &+ 2y   &     &= 3 \\
2x &+      &+ z  &= 5
\end{alignat*}

We can do the same procedure as before &mdash; use the first equation to
eliminate $x$ in the other two equations.  We start by rewriting the first
equation as:

$$x + \frac{1}{\epsilon}y + \frac{1}{\epsilon}z = \frac{6}{\epsilon}$$

then adding or subtracting this from the other two, our system becomes:

\begin{alignat*}{4}
 \epsilon x &+ y                             &+ z                            &= 6 \\
            &+ (2 + \tfrac{1}{\epsilon} )y   &+  \tfrac{1}{\epsilon} z        &= 3 + \tfrac{6}{\epsilon} \\
            &-\tfrac{2}{\epsilon}y           &+ (1 - \tfrac{2}{\epsilon}) z  &= 5 - \tfrac{12}{\epsilon}
\end{alignat*}

Now, as $\epsilon \rightarrow 0$, roundoff error means that

$$\alpha + \frac{\beta}{\epsilon} \approx \frac{\beta}{\epsilon}$$

so, we can rewrite the system approximating the roundoff as:

\begin{alignat*}{4}
 \epsilon x &+ y                             &+ z                            &= 6 \\
            &+ \tfrac{1}{\epsilon} y   &+  \tfrac{1}{\epsilon} z        &\approx \tfrac{6}{\epsilon} \\
            &-\tfrac{2}{\epsilon}y           &- \tfrac{2}{\epsilon} z  &\approx -\tfrac{12}{\epsilon}
\end{alignat*}

But now notice that the last equation is just $-2\times$ the second
equation &mdash; our system has become _singular_, and no longer has
a solution!

The problem arose because we started out the Gaussian elimination with
an equation that had a very small coefficient on $x$.  But we could
reorder the equations, swapping the first and second, and then we
would no longer have this problem.  This is called [partial
pivoting](https://en.wikipedia.org/wiki/Pivot_element#Partial,_rook,_and_complete_pivoting).

In terms of matrix form, partial pivoting means swapping rows of the
matrix such that the element in the column we are eliminating from the
rows below has the largest absolute magnitude.

Let's redo this using pivoting.  To make the notation more compact, we'll use the
augmented matrix form:

$$
\left ( \begin{array}{ccc|c}
       \epsilon  &  1  &  1 & 6 \\
       -1  &  2  &  0 & 3 \\
       2  &  0  &  1 & 5 \end{array} \right )
$$

now we pivot, swapping the first and last rows, sinc the last row has the 
largest magnitude of any entry in the first column:

$$
\left ( \begin{array}{ccc|c}
       2  &  0  &  1 & 5 \\
       -1  &  2  &  0 & 3 \\
       \epsilon  &  1  &  1 & 6 \end{array} \right )
$$

now we do forward elimination, eliminating all the non-zero entries in the first
column beneath row 1:

$$
\left ( \begin{array}{ccc|c}
       2  &  0  &  1 & 5 \\
       0  &  2  &  \tfrac{1}{2} & \tfrac{11}{2} \\
       0  &  1  &  1 - \tfrac{\epsilon}{2} & 6 - \tfrac{5\epsilon}{2} \end{array} \right )
$$

next we can do the rounding, as $\epsilon \rightarrow 0$

$$
\left ( \begin{array}{ccc|c}
       2  &  0  &  1 & 5 \\
       0  &  2  &  \tfrac{1}{2} & \tfrac{11}{2} \\
       0  &  1  &  1 & 6 \end{array} \right )
$$

Note now that this is not singular.  We can continue with forward elimination, removing
all non-zero entries in the second column beneath row 2:

$$
\left ( \begin{array}{ccc|c}
       2  &  0  &  1 & 5 \\
       0  &  2  &  \tfrac{1}{2} & \tfrac{11}{2} \\
       0  &  0  &  \tfrac{3}{4} & \tfrac{13}{4} \end{array} \right )
$$

Now we can do back substitution.  From the last row, we can read off 

$$z = \frac{13}{3}$$

and then putting this into the second row's linear equation, we get

$$2 y + \frac{1}{2} z = 2 y + \frac{13}{6} = \frac{11}{2}$$

or 

$$y = \frac{5}{3}$$

then moving up to the first row, we have:

$$2 x + z = 2 x + \frac{13}{3} = 5$$

so

$$x = \frac{1}{3}$$

We see that pivoting made this system solvable even in the presence of roundoff.

**Note:**
A further refinement on this is [scaled
pivoting](https://en.wikipedia.org/wiki/Pivot_element#Scaled_pivoting)
which is when we first scale each row by its largest element, and then
we consider which row to pivot with.

# Gaussian Elimination

We are now ready to write out the algorithm for [Gaussian elimination](https://en.wikipedia.org/wiki/Gaussian_elimination).
The basic idea is to get the matrix ${\bf A}$ into row-echelon form.
If the matrix is square, and non-singular ($\mbox{det}\{\bf A\} \ne 0$), then a
row-echelon matrix is an upper-trangular matrix.  The process of
transforming the matrix this way is called _forward-elimination_.
Then we solve for ${\bf x}$ by doing _back substitution_.

## Forward Elimination

Consider the following state of our matrix ${\bf A}$ mid-way through the
row-echelon transformation:

$$
\left (
\begin{array}{ccccccc}
  a_{1,1} & a_{1,2} & \cdots & a_{1,k} & a_{1,k+1} & \cdots & a_{1,N} \\
          & a_{2,2} & \cdots & a_{2,k} & a_{2,k+1} & \cdots & a_{2,N} \\
          &         & \ddots  \\
          &         &        & a_{k,k} & a_{k,k+1} & \cdots & a_{k,N} \\
          &         &        & a_{k+1,k} & a_{k+1,k+1} & \cdots & a_{k+1,N} \\
          &         &        & \vdots    & \vdots      &        & \vdots \\
          &         &        & a_{N,k} & a_{N,k+1} & \cdots & a_{N,N} \\
\end{array}
\right )
$$

At this point, the matrix is in row-echelon form down to row $k$, but
the next row, $k+1$ is not, since there is a non-zero element below
the diagonal ($a_{k+1,k}$).

Our next step in forward-elimination
is to operate on all rows $k+1, \ldots, N$ using row $k$ to eliminate
any non-zero entries in column $k$.

The procedure is:

1. Define a coefficient

   $$f_{k+1, k} = \frac{a_{k+1,k}}{a_{k,k}}$$

2. subtract $f_{k+1} \times \{ \mbox{row}~k \}$
   from row $k+1$

3. correct the $k+1$ row of the righthand size vector, ${\bf b}$
   with the same factor

4. Repeat the procedure for all rows $k+2, \ldots, N$, defining new coefficient $f_{k+2,k}, \dots, f_{N,k}$ each time.


Back Substitution
-----------------

At the end of forward elimination, our system looks like:

$$
\left (
\begin{array}{ccccccc}
  a_{1,1} & a_{1,2} & \cdots & a_{1,k} & \cdots   & a_{1,N-1} & a_{1,N} \\
          & a_{2,2} & \cdots & a_{2,k} & \cdots   & a_{2,N-1} & a_{2,N} \\
          &         & \ddots &         &          &           & \vdots  \\
          &         &        & a_{k,k} & \cdots   & a_{k,N-1} & a_{k,N} \\
          &         &        &         & \ddots   &           & \vdots \\
          &         &        &           &        & a_{N-1,N-1} & a_{N-1,N} \\
          &         &        &           &        &             & a_{N,N} \\
\end{array}
\right )
\left (
\begin{array}{c}
 x_1 \\ x_2 \\ \vdots \\ x_k \\ \vdots \\ x_{N-1} \\ x_N
\end{array}
\right )
=
\left (
\begin{array}{c}
 b_1 \\ b_2 \\ \vdots \\ b_k \\ \vdots \\ b_{N-1} \\ b_N
\end{array}
\right )
$$

The last element is simply

$$x_N = \frac{b_N}{a_{N,N}}$$

The second-to-last element is then found by solving:

$$a_{N-1,N-1} x_{N-1} + a_{N-1,N}\underbrace{x_N}_{\mbox{known}} = b_{N-1}$$

giving:

$$x_{N-1} = \frac{b_{N-1} - a_{N-1,N} x_N}{a_{N-1,N-1}}$$

and so forth, with the general case looking like:

$$x_k = \frac{b_k - \sum_{j=k+1}^N a_{k,j} x_j}{a_{k,k}}$$


# Implementing Gaussian Elimination

We start with an implementation of Gaussian elimination as described previously. 

A feature of this implementation is that the input `A` and `b` are changed by this routine, and on output they reflect the row-echelon form.  This is done to save memory.

We will print the steps along the way

In [ ]:
def gauss_elim(A, b, *, quiet=False, pivot=True):
    """ perform gaussian elimination with pivoting, solving A x = b.

        A is an NxN matrix, x and b are an N-element vectors.  Note: A
        and b are changed upon exit to be in upper triangular (row
        echelon) form """

    assert b.ndim == 1, "ERROR: b should be a vector"

    N = len(b)
    assert A.shape == (N, N), "ERROR: A should be square with each dim of same length as b"

    x = np.zeros((N), dtype=A.dtype)

    if not quiet:
        print_Ab(A, b)

    # main loop over rows
    for k in range(N):

        if not quiet:
            print(f"working on row {k}")

        if pivot:
            # find the pivot row based on the size of column k -- only consider
            # the rows >= k (then add k to the index so it is 0-based)
            row_max = np.argmax(np.abs(A[k:, k])) + k

            if row_max != k:
                # swap the row with the largest element in the current column
                # with the current row (pivot) -- do this with b too!
                A[[k, row_max], :] = A[[row_max, k], :]
                b[[k, row_max]] = b[[row_max, k]]
                if not quiet:
                    print("pivoted, updated system:")
                    print_Ab(A, b)

        # do the forward-elimination for all rows below the current
        for i in range(k+1, N):
            coeff = A[i, k] / A[k, k]

            for j in range(k+1, N):
                A[i, j] += -A[k, j] * coeff

            A[i, k] = 0.0
            b[i] += -coeff * b[k]
            
            # check if the row is all zeros -- singular
            if np.abs(A[i, :]).max() == 0:
                if not quiet:
                    print("singular")
                    print_Ab(A, b)
                raise ValueError("matrix is singular")

        if not quiet:
            print_Ab(A, b)

    # back-substitution

    # last solution is easy
    x[N-1] = b[N-1] / A[N-1, N-1]

    for i in reversed(range(N-1)):
        bsum = b[i]
        for j in range(i+1, N):
            bsum += -A[i, j] * x[j]
        x[i] = bsum / A[i, i]

    return x

This routine prints the augmented matrix $({\bf A} | {\bf b})$.

In [ ]:
def print_Ab(A, b):
    """printout the matrix A and vector b in a pretty fashion."""

    N = len(b)

    space = 8*" "
    top_str = "⎧" + N*" {:>7.03f} " + "⎫" + space + "⎧" + " {:6.3f} " + "⎫"
    sid_str = "⎪" + N*" {:>7.03f} " + "⎪" + space + "⎪" + " {:6.3f} " + "⎪"
    bot_str = "⎩" + N*" {:>7.03f} " + "⎭" + space + "⎩" + " {:6.3f} " + "⎭"

    for i in range(N):
        if i == 0:
            pstr = top_str
        elif i == N-1:
            pstr = bot_str
        else:
            pstr = sid_str
        out = tuple(A[i, :]) + (b[i],)
        print(pstr.format(*out))
    print(" ")

Now we can test this out.

**Note:** Be sure to create your arrays as floating point arrays.  If we don't specify, and just include integers in the initialization, then the array will be an integer array, and we will truncate all divisions and won't get the right answer.

In [ ]:
A = np.array([[1, 1, 1],
              [-1, 2, 0],
              [2, 0, 1]], dtype=np.float64)

b = np.array([6, 3, 5], dtype=np.float64)

# since our method changes A and b as it works, we'll send in copies
x = gauss_elim(A.copy(), b.copy())

In [ ]:
print(x)

To ensure that we got the right answer, we can compare ${\bf A}{\bf x}$ and ${\bf b}$.  We can use the python matrix multiplication operator, `@`:

In [ ]:
print(A @ x - b)

## $4\times 4$ example

Let's go through a larger example, with a $4 \times 4$ array.

In [ ]:
A = np.array([[1., 2., 3., 4.],
              [5., 1., 6., -1.],
              [10., 2., 13., -2.],
              [4., 10., -2., -5.]])
b = np.array([3., 2., 4., 1.])

x = gauss_elim(A.copy(), b.copy())

In [ ]:
x

In [ ]:
A @ x - b

## Singular example

changing `A[2, 2]` from `13` to `12` makes the system singular, since row 2 is just double row 1.

In [ ]:
A = np.array([[1., 2., 3., 4.],
              [5., 1., 6., -1.],
              [10., 2., 12., -2.],
              [4., 10., -2., -5.]])
b = np.array([3., 2., 4., 1.])

x = gauss_elim(A, b)

## Roundoff example

Let's try our example with roundoff both with and without pivoting

In [ ]:
eps = 1.e-15

A = np.array([[eps, 1.0, 1.0],
              [-1.0, 2.0, 0.0],
              [2.0, 0.0, 1.0]])
b = np.array([6.0, 3.0, 5.0])

In [ ]:
# With pivoting
x = gauss_elim(A.copy(), b.copy(), quiet=True)
x

In [ ]:
# Without pivoting
x = gauss_elim(A.copy(), b.copy(), quiet=True, pivot=False)
x

Taking $\epsilon \rightarrow 0$, without pivoting, we get a very poor solution.

# Exercises (10 pt)

In the exercises below, we will go through step-by-step another example, to work through the steps to solve a system of linear equations through Gauss-Jordan Elimination:

\begin{eqnarray*}
4x_1 + 3x_2 - 5x_3 &=& 2 \\
-2x_1 - 4x_2 + 5x_3 &=& 5 \\
8x_1 + 8x_2  &=& -3 \\
\end{eqnarray*}

Note, if you are a bit lost, this exercise is heavily based off of this content...\
https://pythonnumericalmethods.studentorg.berkeley.edu/notebooks/chapter14.04-Solutions-to-Systems-of-Linear-Equations.html 

### Exercise 1 (2 pt)

Turn these equations to matrix form ${\bf A} \cdot {\bf x}={\bf b}$. Write the matrix ${\bf A}$, and vector ${\bf b}$, as numpy arrays in the cell below

### Exercise 2 (2 pt)

Get the augmented matrix [A, b], and write the matrix as a numpy array

### Exercise 3 (1 pt)

Now we start to eliminate the elements in the matrix, we do this by choose a **pivot equation**, which is used to eliminate the elements in other equations. Let's choose the first equation as the pivot equation and turn the 2nd row first element to 0. To do this, we can multiply -0.5 for the 1st row (pivot equation) and subtract it from the 2nd row. The multiplier is $f_{2, 1}=-0.5$.

Write the new augmented matrix below, and carry out the operations in python code, (see Gaussian elimination algorithm for examples). Print the resulting matrix

### Exercise 4 (1 pt)

Step 4: Turn the 3rd row first element to 0. We can do something similar, multiply 2 to the 1st row and subtract it from the 3rd row. The multiplier is $f_{3,1}=2$. After doing with matrix operations, write the result below

### Exercise 5 (1 pt)

Turn the 3rd row 2nd element to 0. We can multiple -4/5 for the 2nd row, and add subtract it from the 3rd row. The multiplier is $f_{3,2}=-0.8$. Do the operations, and display the result below

### Exercise 6 (1 pt)

Solve for $x_3$, for which you should get $x_3=-2.2/12=-0.183$. 

### Exercise 7 (1 pt)

Insert $x_3$ to the 2nd row, and solve for $x_2=-2.583$

### Exercise 8 (1 pt)

Insert $x_2$ and $x_3$ to the first equation, and solve $x_1=2.208$. 